In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

# 데이터 준비

In [2]:
df = sns.load_dataset("titanic")

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# 데이터 탐색

In [4]:
df.shape

(891, 15)

In [5]:
df.dtypes

survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object

In [6]:
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [7]:
# 결측치가 많은 deck 열은 삭제, embarked 와 내용이 겹치는 embark_town 열을 삭제
rdf = df.drop(["deck", "embark_town"], axis = 1)

In [8]:
rdf.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'alive', 'alone'],
      dtype='object')

In [9]:
# age 열에 결측치가 있는 모든 행을 삭제(177개 행)
rdf = rdf.dropna(subset = ["age"], axis = 0)

In [10]:
len(rdf), len(df)

(714, 891)

In [11]:
891 - 714

177

In [12]:
rdf["embarked"].unique()

array(['S', 'C', 'Q', nan], dtype=object)

In [16]:
# embarked 열의 NaN 값을 승선도시 중에서 가장 많이 출현한 값(최빈값)으로 치환하기
most_freq = rdf["embarked"].value_counts().idxmax()
most_freq

'S'

In [17]:
rdf.describe(include = "all")

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone
count,714.000000,714.000000,714,714.000000,714.000000,714.000000,714.000000,712,714,714,714,714,714
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,no,True
freq,NaN,NaN,453,NaN,NaN,NaN,NaN,554,355,413,413,424,404
mean,0.406162,2.236695,NaN,29.699118,0.512605,0.431373,34.694514,NaN,NaN,NaN,NaN,NaN,NaN
std,0.491460,0.838250,NaN,14.526497,0.929783,0.853289,52.918930,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,1.000000,NaN,20.125000,0.000000,0.000000,8.050000,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,2.000000,NaN,28.000000,0.000000,0.000000,15.741700,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,1.000000,33.375000,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
rdf["embarked"] = rdf["embarked"].fillna(most_freq)

In [19]:
rdf.isna().sum()

survived      0
pclass        0
sex           0
age           0
sibsp         0
parch         0
fare          0
embarked      0
class         0
who           0
adult_male    0
alive         0
alone         0
dtype: int64

# 분석에 사용할 속성을 선택

In [20]:
rdf.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,no,True


In [21]:
ndf = rdf[["survived", "pclass", "sex", "age", "sibsp", "parch", "embarked"]]

In [22]:
ndf.head()

,survived,pclass,sex,age,sibsp,parch,embarked
0,0,3,male,22.0,1,0,S
1,1,1,female,38.0,1,0,C
2,1,3,female,26.0,0,0,S
3,1,1,female,35.0,1,0,S
4,0,3,male,35.0,0,0,S


In [31]:
# 원핫인코딩 - 범주형 데이터를 인식할 수 있도록 숫자형으로 변환
onehot_sex = pd.get_dummies(ndf["sex"])

In [32]:
onehot_sex.head()

,female,male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


In [33]:
ndf = pd.concat([ndf, onehot_sex], axis = 1)

In [34]:
ndf.head()

,survived,pclass,sex,age,sibsp,parch,embarked,female,male
0,0,3,male,22.0,1,0,S,False,True
1,1,1,female,38.0,1,0,C,True,False
2,1,3,female,26.0,0,0,S,True,False
3,1,1,female,35.0,1,0,S,True,False
4,0,3,male,35.0,0,0,S,False,True


In [39]:
# 승선도시 원핫인코딩
onehot_embarked = pd.get_dummies(ndf["embarked"], prefix = "town")

In [40]:
onehot_embarked.head()

,town_C,town_Q,town_S
0,False,False,True
1,True,False,False
2,False,False,True
3,False,False,True
4,False,False,True


In [41]:
ndf = pd.concat([ndf, onehot_embarked], axis = 1)

In [42]:
ndf.head()

,survived,pclass,sex,age,sibsp,parch,embarked,female,male,town_C,town_Q,town_S
0,0,3,male,22.0,1,0,S,False,True,False,False,True
1,1,1,female,38.0,1,0,C,True,False,True,False,False
2,1,3,female,26.0,0,0,S,True,False,False,False,True
3,1,1,female,35.0,1,0,S,True,False,False,False,True
4,0,3,male,35.0,0,0,S,False,True,False,False,True


In [43]:
ndf = ndf.drop(["sex", "embarked"], axis = 1)

In [44]:
ndf.head()

,survived,pclass,age,sibsp,parch,female,male,town_C,town_Q,town_S
0,0,3,22.0,1,0,False,True,False,False,True
1,1,1,38.0,1,0,True,False,True,False,False
2,1,3,26.0,0,0,True,False,False,False,True
3,1,1,35.0,1,0,True,False,False,False,True
4,0,3,35.0,0,0,False,True,False,False,True


# 데이터셋 분할 - 훈련/테스트

In [45]:
# 변수 선택
x = ndf.drop("survived", axis = 1)
y = ndf["survived"]

In [46]:
# 7 : 3 분할
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, stratify = y,
                                                    random_state = 23)

In [47]:
len(x_train), len(x_test)

(499, 215)

In [79]:
x_train.shape

(499, 9)

# 모델 학습

In [48]:
np.sqrt(len(x_train))

22.338307903688676

In [49]:
# KNN 분류 모형 객체 생성(k = 23 으로 설정)
knn = KNeighborsClassifier(n_neighbors = 23)

In [50]:
# 모델 학습
knn.fit(x_train, y_train)

KNeighborsClassifier(n_neighbors=23)

In [51]:
# test data 예측
y_pred = knn.predict(x_test)

In [52]:
pd.DataFrame({"ans" : y_test, "pred" : y_pred})

,ans,pred
50,0,0
245,0,0
770,0,0
328,1,0
853,1,1
...,...,...
510,1,0
864,0,0
700,1,1
769,0,0


# 모델 평가

In [53]:
# 혼동 행렬
knn_matrix = confusion_matrix(y_test, y_pred)

## 오차 행렬

<img src = "./ML_전체이미지/confusion.png">

- 이진 분류에서 성능 지표로 잘 활용됨

- 분류 모델이 예측을 수행하면서 얼마나 헷갈리고 있는지도 함께 보여주는 지표
    - 이진 분류의 예측 오류가 얼마인지와 더불어 어떠한 유형의 예측 오류가 발생하고 있는지를 함께 나타내는 지표

In [55]:
# 0인 레이블을 0이라 한 경우, 0인 레이블을 1이라 한 경우
# 1인 레이블을 0이라 한 경우, 1인 레이블을 1이라 한 경우
print(knn_matrix)

[[118  10]
 [ 48  39]]


In [56]:
knn_report = classification_report(y_test, y_pred)

In [57]:
print(knn_report)

              precision    recall  f1-score   support

           0       0.71      0.92      0.80       128
           1       0.80      0.45      0.57        87

    accuracy                           0.73       215
   macro avg       0.75      0.69      0.69       215
weighted avg       0.75      0.73      0.71       215



- 정확도
    - 예측 결과와 실제 값이 동일한 건수 / 전체 데이터 수
    - (TN + TP) / (TN + FP + FN + TP)
 
- precision(정밀도)
    - 양성으로 예측한 데이터 중 실제로 양성인 데이터의 비율
    - TP / (FP + TP)
    - 정밀도가 더 중요한 지표인 경우는 양성으로 분류된 데이터가 실제로는 양성이 아니라면 업무상 큰 영향이 발생하는 경우
        - 예) 스팸메일 판단 모델 : 일반 메일을 스팸메일로 잘못 분류하면 메일을 아예 받지 못할 수 있어 업무에 차질이 생김
     
- recall(재현율, 민감도)
    - 실제 양성인 데이터 중 양성으로 예측한 비율
    - TP / (FN + TP)
    - 재현율이 중요 지표인 경우는 실제 양성 데이터를 음성으로 잘못 판단하게 되면 업무상 큰 영향이 발생하는 경우
        - 예) 암 판단 모델 : 암환자를 음성으로 잘못 판단한다면 생명을 앗아갈 정도로 심각한 문제이기 때문
     
- specificity(특이도)
    - 실제 음성인 데이터 중 음성으로 예측한 비율
    - TN / (TN + FP)
 
- 코로나 검사키트의 경우
    - 민감도 90% 이상
        - 실제 양성인 사람이 코로나 검사를 하면 양성으로 예측될 확률 90% 이상
     
    - 특이도 99% 이상
        - 실제 음성인 사람이 코로나 검사를 하면 음성으로 예측될 확률 99% 이상

# 모델 고도화

In [58]:
y.value_counts()

survived
0    424
1    290
Name: count, dtype: int64

In [59]:
# 스케일링
ss = StandardScaler()

In [60]:
scaled_train = ss.fit_transform(x_train)
scaled_test = ss.transform(x_test)

In [61]:
knn = KNeighborsClassifier(n_neighbors = 23)

In [62]:
knn.fit(scaled_train, y_train)

KNeighborsClassifier(n_neighbors=23)

In [63]:
y_pred = knn.predict(scaled_test)

In [64]:
pd.DataFrame({"ans" : y_test, "pred" : y_pred})

,ans,pred
50,0,0
245,0,0
770,0,0
328,1,1
853,1,1
...,...,...
510,1,0
864,0,0
700,1,1
769,0,0


In [65]:
knn_matrix = confusion_matrix(y_test, y_pred)
print(knn_matrix)

[[115  13]
 [ 33  54]]


In [66]:
knn_report = classification_report(y_test, y_pred)
print(knn_report)

              precision    recall  f1-score   support

           0       0.78      0.90      0.83       128
           1       0.81      0.62      0.70        87

    accuracy                           0.79       215
   macro avg       0.79      0.76      0.77       215
weighted avg       0.79      0.79      0.78       215



In [75]:
# K값을 11로 수정
knn = KNeighborsClassifier(n_neighbors = 11)

In [76]:
knn.fit(scaled_train, y_train)

KNeighborsClassifier(n_neighbors=11)

In [77]:
y_pred = knn.predict(scaled_test)

In [78]:
knn_report = classification_report(y_test, y_pred)
print(knn_report)

              precision    recall  f1-score   support

           0       0.76      0.88      0.82       128
           1       0.78      0.60      0.68        87

    accuracy                           0.77       215
   macro avg       0.77      0.74      0.75       215
weighted avg       0.77      0.77      0.76       215

